# 13 — Validate Phenotype-Stratified Antibiogram Reporting

**Project:** Clinical Microbiology Analytics Lakehouse  
**Source:** Stanford ARMD (public de-identified data)  
**Warehouse:** `workspace.micro_dev`

This notebook converts the five queries from `validation_queries.sql` into executable, separately labeled Databricks SQL cells. It evaluates the phenotype-stratified susceptibility mart alongside the independently defined MDRO mart.

## Prerequisites and reporting rules

Run this notebook **only after** applying the phenotype-stratified dbt changes and completing `dbt build` successfully. For the first regression comparison, keep the reporting window in **full-release mode**.

- Ordinary (`base`) and resistant reporting categories independently select their first qualifying culture per patient, organism and specimen cohort. The same patient may appear in more than one category.
- Overall *S. aureus* uses the first culture regardless of phenotype, while MRSA/MSSA are selected separately.
- CRE and MDR *P. aeruginosa* are excluded from the main susceptibility mart and summarized separately by `mart_mdro`.
- These are descriptive research summaries based on source-reported AST, not a guideline-certified clinical antibiogram.

**Execution:** Use the VS Code notebook kernel configured for Databricks Connect with `%sql` support. Each cell contains **one SQL statement**, and `%sql` is the first line. Results have not been executed or saved in this notebook.

### 1. Inspect reporting categories

**Purpose:** List organism/reporting-phenotype categories actually appearing in `mart_susceptibility`, including their row counts and largest eligible first-culture population.

**Interpretation:** The main susceptibility mart should not contain the CRE or MDR *P. aeruginosa* categories. The `overall` category should be reserved for *Staphylococcus aureus*; other organism populations are stratified into `base` and qualifying phenotype categories. This is an **inventory query**, not a standalone assertion; it only displays categories named in its SQL filter.

In [4]:
%sql
select reporting_phenotype, organism_reporting_group,
       count(*) as mart_rows,
       max(eligible_count) as maximum_eligible_count
from workspace.micro_dev.mart_susceptibility
where reporting_phenotype in ('overall','esce_r','base',
  'oxacillin_resistant','oxacillin_susceptible','vancomycin_resistant',
  'cre','mdr_p_aeruginosa')
group by reporting_phenotype, organism_reporting_group
order by reporting_phenotype, organism_reporting_group;

,reporting_phenotype,organism_reporting_group,mart_rows,maximum_eligible_count
0,base,Citrobacter freundii complex,18,860
1,base,Citrobacter koseri,23,1004
2,base,Coagulase-negative staphylococci (unspeciated),3,1247
3,base,Enterobacter cloacae complex,12,927
4,base,Enterococcus faecalis,15,867
5,base,Enterococcus faecium,15,135
6,base,Enterococcus species (unspeciated),2,7354
7,base,Escherichia coli,21,34714
8,base,Klebsiella aerogenes,20,826
9,base,Klebsiella oxytoca,18,962


### 2. Quantify overlap across separate phenotype reporting populations

**Purpose:** Count distinct patients represented in the `base`, ESCE-R, and VRE reporting categories for each cohort and organism.

**Interpretation:** A positive `patients_in_both_categories` is **expected and allowed**. A patient can contribute one first culture to `base` and another first qualifying culture to ESCE-R or VRE. Do not sum these populations to estimate unique patients. This query does not measure MRSA/MSSA overlap.

In [5]:
%sql
with labels as (
    select source_version, cohort_id, organism_reporting_group,
           patient_key,
           max(case when reporting_phenotype='base' then 1 else 0 end) as base_patient,
           max(case when reporting_phenotype='esce_r' then 1 else 0 end) as esce_patient,
           max(case when reporting_phenotype='vancomycin_resistant' then 1 else 0 end) as vre_patient
    from workspace.micro_dev.int_first_stratified_reporting_culture
    group by source_version, cohort_id, organism_reporting_group, patient_key
)
select cohort_id, organism_reporting_group,
       sum(base_patient) as base_patients,
       sum(esce_patient) as esce_r_patients,
       sum(vre_patient) as vre_patients,
       sum(case when base_patient=1 and (esce_patient=1 or vre_patient=1)
           then 1 else 0 end) as patients_in_both_categories
from labels
group by cohort_id, organism_reporting_group
having sum(esce_patient)+sum(vre_patient)>0
order by patients_in_both_categories desc, cohort_id, organism_reporting_group;

,cohort_id,organism_reporting_group,base_patients,esce_r_patients,vre_patients,patients_in_both_categories
0,urine,Escherichia coli,34714,3500,0,1119
1,urine,Klebsiella pneumoniae,6114,393,0,96
2,blood,Escherichia coli,2565,495,0,41
3,urine,Enterobacter cloacae complex,927,184,0,28
4,urine,Klebsiella aerogenes,826,103,0,11
5,urine,Klebsiella oxytoca,962,66,0,11
6,respiratory,Escherichia coli,320,67,0,7
7,blood,Enterococcus faecium,135,0,139,5
8,urine,Enterococcus faecalis,867,0,8,4
9,urine,Enterococcus faecium,125,0,77,4


### 3. Check *Staphylococcus aureus* populations

**Purpose:** Report the eligible first-culture denominator separately for `overall`, `oxacillin_resistant` (MRSA), and `oxacillin_susceptible` (MSSA) within each specimen cohort.

**Interpretation:** `overall` is the first *S. aureus* culture irrespective of phenotype; MRSA and MSSA are selected independently as first qualifying cultures. The query uses `max(eligible_count)` because the population count is repeated across the approved antibiotics in each category.

In [6]:
%sql
select cohort_id, reporting_phenotype, max(eligible_count) as n_first_cultures
from workspace.micro_dev.mart_susceptibility
where organism_reporting_group='Staphylococcus aureus'
group by cohort_id, reporting_phenotype
order by cohort_id, reporting_phenotype;

,cohort_id,reporting_phenotype,n_first_cultures
0,blood,overall,1737
1,blood,oxacillin_resistant,546
2,blood,oxacillin_susceptible,1206
3,respiratory,overall,3430
4,respiratory,oxacillin_resistant,618
5,respiratory,oxacillin_susceptible,2931
6,urine,overall,517
7,urine,oxacillin_resistant,174
8,urine,oxacillin_susceptible,331


### 4. Compare MDRO surveillance totals

**Purpose:** Summarize `mart_mdro` row counts, eligible patient counts, and phenotype-positive patient counts by cohort and MDRO definition.

**Interpretation:** In **full-release mode**, compare these numbers with a saved, same-release baseline. They should remain unchanged by susceptibility-category restructuring. If a restricted reporting window is active, differences are expected. The MDRO mart has a separate any-positive-across-observations methodology.

In [7]:
%sql
-- relative to the last known validated baseline.
select mdro_phenotype, cohort_id, count(*) as rows,
       sum(denominator_count) as patients,
       sum(positive_count) as positive_patients
from workspace.micro_dev.mart_mdro
group by mdro_phenotype, cohort_id
order by mdro_phenotype, cohort_id;

,mdro_phenotype,cohort_id,rows,patients,positive_patients
0,CRE,blood,10,5203,27
1,CRE,respiratory,10,1911,17
2,CRE,urine,11,52732,130
3,MDR Pseudomonas aeruginosa,blood,1,384,24
4,MDR Pseudomonas aeruginosa,respiratory,1,1889,254
5,MDR Pseudomonas aeruginosa,urine,1,1956,72


### 5. Inspect shared reporting AST reconciliation

**Purpose:** Inspect the `ast_scope` categories, endpoint volumes, reporting source groups, and conflict counts in `int_culture_reporting_susceptibility`.

**Interpretation:** This descriptive check makes the provenance of selected AST observations visible. Review whether the scopes reflect the new phenotype-stratified design and whether conflicts are retained rather than silently assigned a result.

In [8]:
%sql
select ast_scope, count(*) as endpoint_rows,
       count(distinct reporting_source_key) as culture_groups,
       sum(case when resolution_status='conflict' then 1 else 0 end) as conflicting_endpoints
from workspace.micro_dev.int_culture_reporting_susceptibility
group by ast_scope;

,ast_scope,endpoint_rows,culture_groups,conflicting_endpoints
0,overall,98929,9906,771
1,stratified,1523124,123828,9557


In [9]:
%sql
select
    reporting_phenotype,
    count(*) as mart_rows,
    sum(case
        when display_status = 'display' then 1
        else 0
    end) as displayable_rows
from workspace.micro_dev.mart_susceptibility
group by reporting_phenotype
order by reporting_phenotype;

,reporting_phenotype,mart_rows,displayable_rows
0,base,287,264
1,esce_r,89,53
2,overall,18,18
3,oxacillin_resistant,18,18
4,oxacillin_susceptible,18,18
5,vancomycin_resistant,25,10


12-month reporting coverage

In [10]:
%sql
select
    cohort_id,
    reporting_phenotype,
    count(*) as total_pairs,
    sum(
        case
            when display_status = 'display' then 1
            else 0
        end
    ) as displayable_pairs,
    sum(
        case
            when display_status = 'insufficient_data' then 1
            else 0
        end
    ) as insufficient_data_pairs,
    round(
        100.0 * sum(
            case
                when display_status = 'display' then 1
                else 0
            end
        ) / nullif(count(*), 0),
        1
    ) as pct_displayable
from workspace.micro_dev.mart_susceptibility
group by
    cohort_id,
    reporting_phenotype
order by
    cohort_id,
    reporting_phenotype;

,cohort_id,reporting_phenotype,total_pairs,displayable_pairs,insufficient_data_pairs,pct_displayable
0,blood,base,100,35,65,35.0
1,blood,esce_r,30,6,24,20.0
2,blood,overall,6,6,0,100.0
3,blood,oxacillin_resistant,6,6,0,100.0
4,blood,oxacillin_susceptible,6,6,0,100.0
5,blood,vancomycin_resistant,10,0,10,0.0
6,respiratory,base,75,14,61,18.7
7,respiratory,esce_r,30,0,30,0.0
8,respiratory,overall,6,6,0,100.0
9,respiratory,oxacillin_resistant,6,6,0,100.0


In [11]:
%sql
select
    analysis_window_start,
    analysis_window_end_exclusive,
    cohort_id,
    reporting_phenotype,
    count(*) as total_pairs,
    sum(case
        when display_status = 'display' then 1
        else 0
    end) as displayable_pairs,
    sum(case
        when display_status = 'insufficient_data' then 1
        else 0
    end) as insufficient_data_pairs,
    round(
        100.0 * sum(case
            when display_status = 'display' then 1
            else 0
        end) / nullif(count(*), 0),
        1
    ) as pct_displayable
from workspace.micro_dev.mart_susceptibility
group by
    analysis_window_start,
    analysis_window_end_exclusive,
    cohort_id,
    reporting_phenotype
order by
    cohort_id,
    reporting_phenotype;

,analysis_window_start,analysis_window_end_exclusive,cohort_id,reporting_phenotype,total_pairs,displayable_pairs,insufficient_data_pairs,pct_displayable
0,2023-01-01,2024-01-01,blood,base,100,35,65,35.0
1,2023-01-01,2024-01-01,blood,esce_r,30,6,24,20.0
2,2023-01-01,2024-01-01,blood,overall,6,6,0,100.0
3,2023-01-01,2024-01-01,blood,oxacillin_resistant,6,6,0,100.0
4,2023-01-01,2024-01-01,blood,oxacillin_susceptible,6,6,0,100.0
5,2023-01-01,2024-01-01,blood,vancomycin_resistant,10,0,10,0.0
6,2023-01-01,2024-01-01,respiratory,base,75,14,61,18.7
7,2023-01-01,2024-01-01,respiratory,esce_r,30,0,30,0.0
8,2023-01-01,2024-01-01,respiratory,overall,6,6,0,100.0
9,2023-01-01,2024-01-01,respiratory,oxacillin_resistant,6,6,0,100.0


## Review checklist

After running all five cells, verify the reporting category inventory, review the expected overlaps among independently selected phenotype populations, examine the *S. aureus* overall/MRSA/MSSA counts, compare MDRO totals with the full-release baseline, and inspect the AST-scope reconciliation summaries.

These queries complement dbt's automated tests; they are **not** a substitute for the full build or separate row-level denominator and approved-pair assertions. Record any unexpected categories or counts for investigation before exporting the reporting mart.